In [20]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from matplotlib import rcParams
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
df = pd.read_csv('merged_Results_Spillover_Random_TOOv2.txt', sep='\t')

# Extract reference and max genes from matrix names
df['Reference'] = df['Matrix'].apply(lambda x: x.split('_')[0])
df['MaxGenes'] = df['Matrix'].apply(lambda x: x.split('_')[-1])

# Create output directory
output_dir = 'Results_Spillover_Grouped'
os.makedirs(output_dir, exist_ok=True)

# --- Global font sizes  ---
rcParams.update({
    "axes.titlesize": 13,     # plot title
    "axes.labelsize": 12,     # x/y labels
    "xtick.labelsize": 10,    # x tick labels
    "ytick.labelsize": 10,    # y tick labels
    "legend.title_fontsize": 11,
    "legend.fontsize": 10
})

SP = "__SP__"  # spacer prefix
ALL_GENES_COLOR = (0.25, 0.25, 0.25)  # dark grey for All Genes

for reference in df['Reference'].unique():
    subset = df[df['Reference'] == reference]

    special_methods = ['BayesPrism', 'MuSiC']
    normal_df = subset[~subset['Method'].isin(special_methods)].copy()
    special_df = subset[subset['Method'].isin(special_methods)].copy()

    methods = subset['Method'].unique().tolist()
    methods = [m for m in methods if m not in special_methods + ['nuSVR']]
    methods.sort()
    custom_order = [m for m in special_methods if m in subset['Method'].unique()]
    if 'nuSVR' in subset['Method'].unique():
        custom_order += ['nuSVR']
    custom_order += methods

    plt.figure(figsize=(10, 3))

    normal_df['XLabel'] = normal_df.apply(lambda r: f"{r['Method']}__{r['MaxGenes']}", axis=1)
    special_df['XLabel'] = special_df['Method']

    # --- order with an invisible spacer BEFORE the 500 violin of each normal method and before MuSiC
    x_order = []
    for m in custom_order:
        if m in special_methods:
            x_order.append(m)
            if m == "BayesPrism":
                x_order.append(f"{SP}{m}_to_MuSiC")  # spacer between special methods
        else:
            x_order.append(f"{SP}{m}")              # spacer before 500
            x_order += [f"{m}__{g}" for g in maxgenes_order]

    # palette (add transparent color for spacers)
    g2c = dict(zip(maxgenes_order, cb_palette))
    transparent = (0, 0, 0, 0)
    palette_map = {}
    for m in custom_order:
        if m in special_methods:
            palette_map[m] = ALL_GENES_COLOR
            if m == "BayesPrism":
                palette_map[f"{SP}{m}_to_MuSiC"] = transparent
        else:
            palette_map[f"{SP}{m}"] = transparent
            for g in maxgenes_order:
                palette_map[f"{m}__{g}"] = g2c[g]

    plot_df = pd.concat([normal_df, special_df], ignore_index=True)

    ax = sns.violinplot(
        data=plot_df,
        x='XLabel',
        y='Average',
        order=x_order,
        palette=palette_map,
        inner='box',
        density_norm='width',
        width=0.80,  # wide violins
        cut=0
    )

    # --- Apply hatch ONLY to the All Genes violins (BayesPrism, MuSiC)
    label_to_x = {lab: idx for idx, lab in enumerate(x_order)}
    special_x = {label_to_x[m] for m in ['BayesPrism', 'MuSiC'] if m in label_to_x}

    for coll in [c for c in ax.collections if hasattr(c, "get_paths")]:
        try:
            verts = coll.get_paths()[0].vertices
        except Exception:
            continue
        xmean = verts[:, 0].mean()
        if round(xmean) in special_x:
            coll.set_facecolor(ALL_GENES_COLOR)
            coll.set_hatch('//')
            coll.set_edgecolor('black')

    # center method labels (skip spacers)
    xticks, xticklabels = [], []
    for m in custom_order:
        if m in special_methods:
            pos = x_order.index(m)
            xticks.append(pos); xticklabels.append(m)
        else:
            trio = [x_order.index(f"{m}__{g}") for g in maxgenes_order]
            xticks.append(sum(trio)/len(trio)); xticklabels.append(m)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=30, ha='right')

    # headroom + compact margins
    ax.set_ylim(1, 7.5)
    ax.margins(x=0.01, y=0.05)

    # titles/labels (fonts already set via rcParams)
    ax.set_title(f"MAE for {reference} reference")
    ax.set_xlabel('Deconvolution Method')
    ax.set_ylabel('Mean Absolute Error')

    # legend + mean dots
    MEAN_DOT_SIZE = 35
    legend_handles = [
        Patch(facecolor=ALL_GENES_COLOR, edgecolor='black', hatch='//', label='All Genes')
    ]
    legend_handles += [Patch(facecolor=g2c[g], edgecolor='black', label=g) for g in maxgenes_order]

    means = plot_df.groupby('XLabel')['Average'].mean()
    for xpos, label in enumerate(x_order):
        if label.startswith(SP):
            continue
        if label in means:
            ax.scatter(xpos, means[label], color='red', s=MEAN_DOT_SIZE, zorder=3,
                       marker='o', edgecolor='black')

    legend_handles += [Line2D([0], [0], marker='o', color='red', label='Mean',
                              markerfacecolor='red', markeredgecolor='black',
                              markersize=12, linestyle='')]

    ax.legend(handles=legend_handles, title='Max Signatures',
              bbox_to_anchor=(1.02, 1), loc='upper left')

    fig = plt.gcf()  # get current figure
    filename = f'Violin_Random_{reference}_all_methods.svg'.replace(' ', '_')
    fig.savefig(os.path.join(output_dir, filename), bbox_inches="tight")
    plt.close(fig)


/tmp/ipykernel_813478/1364805485.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/tmp/ipykernel_813478/1364805485.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/tmp/ipykernel_813478/1364805485.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(


In [21]:
# Load the data
df = pd.read_csv('merged_Results_Spillover_Uniform_TOOv2.txt', sep='\t')

# Extract reference and max genes from matrix names
df['Reference'] = df['Matrix'].apply(lambda x: x.split('_')[0])
df['MaxGenes'] = df['Matrix'].apply(lambda x: x.split('_')[-1])

# Create output directory
output_dir = 'Results_Spillover_Grouped'
os.makedirs(output_dir, exist_ok=True)


SP = "__SP__"  # spacer prefix
ALL_GENES_COLOR = (0.25, 0.25, 0.25)  # dark grey for All Genes

for reference in df['Reference'].unique():
    subset = df[df['Reference'] == reference]

    special_methods = ['BayesPrism', 'MuSiC']
    normal_df = subset[~subset['Method'].isin(special_methods)].copy()
    special_df = subset[subset['Method'].isin(special_methods)].copy()

    methods = subset['Method'].unique().tolist()
    methods = [m for m in methods if m not in special_methods + ['nuSVR']]
    methods.sort()
    custom_order = [m for m in special_methods if m in subset['Method'].unique()]
    if 'nuSVR' in subset['Method'].unique():
        custom_order += ['nuSVR']
    custom_order += methods

    plt.figure(figsize=(10, 3))

    normal_df['XLabel'] = normal_df.apply(lambda r: f"{r['Method']}__{r['MaxGenes']}", axis=1)
    special_df['XLabel'] = special_df['Method']

    # --- order with an invisible spacer BEFORE the 500 violin of each normal method and before MuSiC
    x_order = []
    for m in custom_order:
        if m in special_methods:
            x_order.append(m)
            if m == "BayesPrism":
                x_order.append(f"{SP}{m}_to_MuSiC")  # spacer between special methods
        else:
            x_order.append(f"{SP}{m}")              # spacer before 500
            x_order += [f"{m}__{g}" for g in maxgenes_order]

    # palette (add transparent color for spacers)
    g2c = dict(zip(maxgenes_order, cb_palette))
    transparent = (0, 0, 0, 0)
    palette_map = {}
    for m in custom_order:
        if m in special_methods:
            palette_map[m] = ALL_GENES_COLOR
            if m == "BayesPrism":
                palette_map[f"{SP}{m}_to_MuSiC"] = transparent
        else:
            palette_map[f"{SP}{m}"] = transparent
            for g in maxgenes_order:
                palette_map[f"{m}__{g}"] = g2c[g]

    plot_df = pd.concat([normal_df, special_df], ignore_index=True)

    ax = sns.violinplot(
        data=plot_df,
        x='XLabel',
        y='Average',
        order=x_order,
        palette=palette_map,
        inner='box',
        density_norm='width',
        width=0.80,  # wide violins
        cut=0
    )

    # --- Apply hatch ONLY to the All Genes violins (BayesPrism, MuSiC)
    label_to_x = {lab: idx for idx, lab in enumerate(x_order)}
    special_x = {label_to_x[m] for m in ['BayesPrism', 'MuSiC'] if m in label_to_x}

    for coll in [c for c in ax.collections if hasattr(c, "get_paths")]:
        try:
            verts = coll.get_paths()[0].vertices
        except Exception:
            continue
        xmean = verts[:, 0].mean()
        if round(xmean) in special_x:
            coll.set_facecolor(ALL_GENES_COLOR)
            coll.set_hatch('//')
            coll.set_edgecolor('black')

    # center method labels (skip spacers)
    xticks, xticklabels = [], []
    for m in custom_order:
        if m in special_methods:
            pos = x_order.index(m)
            xticks.append(pos); xticklabels.append(m)
        else:
            trio = [x_order.index(f"{m}__{g}") for g in maxgenes_order]
            xticks.append(sum(trio)/len(trio)); xticklabels.append(m)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=30, ha='right')

    # headroom + compact margins
    ax.set_ylim(1, 7.5)
    ax.margins(x=0.01, y=0.05)

    # titles/labels (fonts already set via rcParams)
    ax.set_title(f"MAE for {reference} reference")
    ax.set_xlabel('Deconvolution Method')
    ax.set_ylabel('Mean Absolute Error')

    # legend + mean dots
    MEAN_DOT_SIZE = 35
    legend_handles = [
        Patch(facecolor=ALL_GENES_COLOR, edgecolor='black', hatch='//', label='All Genes')
    ]
    legend_handles += [Patch(facecolor=g2c[g], edgecolor='black', label=g) for g in maxgenes_order]

    means = plot_df.groupby('XLabel')['Average'].mean()
    for xpos, label in enumerate(x_order):
        if label.startswith(SP):
            continue
        if label in means:
            ax.scatter(xpos, means[label], color='red', s=MEAN_DOT_SIZE, zorder=3,
                       marker='o', edgecolor='black')

    legend_handles += [Line2D([0], [0], marker='o', color='red', label='Mean',
                              markerfacecolor='red', markeredgecolor='black',
                              markersize=12, linestyle='')]

    ax.legend(handles=legend_handles, title='Max Signatures',
              bbox_to_anchor=(1.02, 1), loc='upper left')

    fig = plt.gcf()  # get current figure
    filename = f'Violin_Uniform_{reference}_all_methods.svg'.replace(' ', '_')
    fig.savefig(os.path.join(output_dir, filename), bbox_inches="tight")
    plt.close(fig)



/tmp/ipykernel_813478/755066188.py:63: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/tmp/ipykernel_813478/755066188.py:63: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(
/tmp/ipykernel_813478/755066188.py:63: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.violinplot(


In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === LOAD DATA ===
file_path = 'merged_Results_Spillover_Random_TOOv2.txt'
df = pd.read_csv(file_path, sep='\t')

# === DEFINE TISSUE COLUMNS ===
tissue_cols = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast', 'Colon', 'Esophagus', 'FemaleReproductive',
    'Fibroblasts', 'Heart', 'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal', 'NerveTibial', 'Pancreas',
    'Pituitary', 'Prostate', 'SalivaryGland', 'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Whole blood'
]

# === SPLIT AND FILTER DATA ===
df_common = df[(df['Matrix'] == '2Median_300_500') & (df['Method'] != 'ReDeconv')]
df_redeconv = df[(df['Matrix'] == 'Sampling10_300_500') & (df['Method'] == 'ReDeconv')]
df_filtered = pd.concat([df_common, df_redeconv], ignore_index=True)

# === GROUP BY METHOD AND AVERAGE SPILLOVER ===
heatmap_df = df_filtered.groupby('Method')[tissue_cols].mean()

# === RENAME COLUMNS ===
rename_map = {
    "SmallIntestine": "Small intestine",
    "SalivaryGland": "Salivary gland",
    "FemaleReproductive": "Female reproductive",
    "NerveTibial": "Tibial nerve",
    "MuscleSkeletal": "Skeletal muscle",
    "Pituitary": "Pituitary gland"
}
heatmap_df = heatmap_df.rename(columns=rename_map)
# === SORT TISSUES ALPHABETICALLY ===
heatmap_df = heatmap_df.reindex(sorted(heatmap_df.columns), axis=1)

# === REORDER ROWS ===
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
heatmap_df = heatmap_df.reindex(method_order)

# === PLOT HEATMAP ===
fig, ax = plt.subplots(figsize=(10, 5), dpi=600, constrained_layout=True)
ax = sns.heatmap(
    heatmap_df,
    cmap='viridis_r',                      # keep consistent colormap style
#    annot=True,
#    fmt=".2f",
#    annot_kws={"size": 6},
    cbar_kws={'label': 'Mean Absolute Error', 'shrink': 0.9},
    linewidths=0,
    linecolor='white'
)

# === Tick styling (consistent with other plots) ===
ax.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=4, width=0.6
)
ax.tick_params(
    axis='y', which='both', left=True, right=False, labelleft=True,
    length=4, width=0.6
)

# === Colorbar styling ===
cbar = ax.collections[0].colorbar
cbar.ax.set_ylabel("Mean Absolute Error", fontsize=13, rotation=270, labelpad=20)   
cbar.ax.tick_params(labelsize=11, width=0.8, length=4)

# === Axes styling ===
plt.ylabel('Deconvolution Tool', fontsize=14, labelpad=10)
plt.xlabel('Tissue', fontsize=14, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

fig.savefig("Heatmap_Random_Spillover_PerTissue_Final_Clean.svg", bbox_inches="tight")
plt.close(fig)


In [4]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === LOAD DATA ===
file_path = 'merged_Results_Spillover_Absent_Random_TOOv2.txt'
df = pd.read_csv(file_path, sep='\t')

# === DEFINE TISSUE COLUMNS ===
tissue_cols = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast', 'Colon', 'Esophagus', 'FemaleReproductive',
    'Fibroblasts', 'Heart', 'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal', 'NerveTibial', 'Pancreas',
    'Pituitary', 'Prostate', 'SalivaryGland', 'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Whole blood'
]

# === SPLIT AND FILTER DATA ===
df_common = df[(df['Matrix'] == '2Median_300_500') & (df['Method'] != 'ReDeconv')]
df_redeconv = df[(df['Matrix'] == 'Sampling10_300_500') & (df['Method'] == 'ReDeconv')]
df_filtered = pd.concat([df_common, df_redeconv], ignore_index=True)

# === GROUP BY METHOD AND AVERAGE SPILLOVER ===
heatmap_df = df_filtered.groupby('Method')[tissue_cols].mean()

# === RENAME COLUMNS ===
rename_map = {
    "SmallIntestine": "Small intestine",
    "SalivaryGland": "Salivary gland",
    "FemaleReproductive": "Female reproductive",
    "NerveTibial": "Tibial nerve",
    "MuscleSkeletal": "Skeletal muscle",
    "Pituitary": "Pituitary gland"
}
heatmap_df = heatmap_df.rename(columns=rename_map)

# === SORT TISSUES ALPHABETICALLY ===
heatmap_df = heatmap_df.reindex(sorted(heatmap_df.columns), axis=1)

# === REORDER ROWS ===
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
heatmap_df = heatmap_df.reindex(method_order)

# === PLOT HEATMAP ===
fig, ax = plt.subplots(figsize=(10, 5), dpi=600, constrained_layout=True)
ax = sns.heatmap(
    heatmap_df,
    cmap='viridis_r',                      # keep consistent colormap style
#    annot=True,
#    fmt=".2f",
#    annot_kws={"size": 6},
    cbar_kws={'label': 'Mean Absolute Error', 'shrink': 0.9},
    linewidths=0,
    linecolor='white'
)

# === Tick styling (consistent with other plots) ===
ax.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=4, width=0.6
)
ax.tick_params(
    axis='y', which='both', left=True, right=False, labelleft=True,
    length=4, width=0.6
)

# === Colorbar styling ===
cbar = ax.collections[0].colorbar
cbar.ax.set_ylabel("Mean Absolute Error", fontsize=11, rotation=270, labelpad=20)   
cbar.ax.tick_params(labelsize=10, width=0.8, length=4)

# === Axes styling ===
plt.ylabel('Deconvolution Tool', fontsize=11)
plt.xlabel('Tissue', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)

fig.savefig("Heatmap_Random_Spillover_PerTissue_Absent_Clean.svg", bbox_inches="tight")
plt.close(fig)
